# Simple research pipeline (OpenAI Agents SDK)

This notebook follows a pattern seen across many community submissions:

1. **Plan** structured web searches from a question (Pydantic output).
2. **Search** with the hosted `WebSearchTool`, one query at a time.
3. **Write** a short markdown report from the collected notes.

Set `OPENAI_API_KEY` in a `.env` file (here or repo root), then run cells in order.

In [ ]:
%pip install -q openai-agents python-dotenv pydantic

In [ ]:
from __future__ import annotations

import os

from dotenv import load_dotenv
from pydantic import BaseModel, Field

from agents import Agent, Runner, WebSearchTool, set_tracing_disabled
from agents.model_settings import ModelSettings

In [ ]:
load_dotenv(override=True)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file.")

set_tracing_disabled(True)

MODEL = "gpt-4o-mini"
NUM_SEARCHES = 3

In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(description="Why this search helps answer the question.")
    query: str = Field(description="Search query string.")


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description=f"Exactly {NUM_SEARCHES} focused web searches."
    )


class ReportData(BaseModel):
    markdown_report: str = Field(description="Short markdown report with headings.")


web_search = WebSearchTool(search_context_size="low")

In [ ]:
planner = Agent(
    name="Planner",
    instructions=(
        f"Given a research question, propose exactly {NUM_SEARCHES} distinct web searches. "
        "Each item must include a brief reason and a concrete query."
    ),
    model=MODEL,
    output_type=WebSearchPlan,
)

searcher = Agent(
    name="Searcher",
    instructions=(
        "Use the web_search tool at least once. Given a query and reason, "
        "return a tight bullet summary (under 200 words) of relevant facts only."
    ),
    tools=[web_search],
    model=MODEL,
    model_settings=ModelSettings(tool_choice="required"),
)

writer = Agent(
    name="Writer",
    instructions=(
        "You receive the original question and raw search notes. "
        "Produce a clear markdown report: brief intro, sections with subheadings, "
        "and a short conclusion. Do not invent sources beyond the notes."
    ),
    model=MODEL,
    output_type=ReportData,
)

In [ ]:
async def research(question: str) -> str:
    plan_result = await Runner.run(planner, question)
    plan = plan_result.final_output_as(WebSearchPlan)

    notes: list[str] = []
    for item in plan.searches:
        prompt = f"Query: {item.query}\nReason: {item.reason}"
        search_result = await Runner.run(searcher, prompt)
        notes.append(str(search_result.final_output))

    combined = (
        f"Question:\n{question}\n\n"
        f"Search notes ({len(notes)}):\n" + "\n---\n".join(notes)
    )
    report_result = await Runner.run(writer, combined)
    report = report_result.final_output_as(ReportData)
    return report.markdown_report

In [ ]:
TOPIC = "What are practical ways teams evaluate retrieval quality for RAG systems in 2025?"

report = await research(TOPIC)
print(report)